# Requirements

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import nltk
import emoji
import spacy
import string
import unicodedata
import datetime
import random
from sklearn.utils import shuffle
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings("ignore")

portuguese_stopwords = nltk.corpus.stopwords.words('portuguese')

# Get Data

In [2]:
df = pd.read_csv('df_topicos_tv_pt_br.csv', sep = ';', encoding = 'utf-8')

# Data Prep Functions

In [3]:
def preprocess_data(data, 
                    columns,
                    null = True):
    
    df = data[columns]
    
    if null:
        df = df.dropna(subset = ['COMMENT_TEXT'])
    
    return df


def preprocess_text(text, 
                    remove_stop = True, 
                    stem_words = False, 
                    remove_mentions_hashtags = True
                   ):
    """
    eg:
    input: preprocess_text("@water #dream hi hello where are you going be there tomorrow happening happen happens",  
    stem_words = True) 
    output: ['tomorrow', 'happen', 'go', 'hello']
    """

    # Remove emojis
    emoji_pattern = re.compile("[" "\U0001F1E0-\U0001F6FF" "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r"", text)
    text = "".join([x for x in text if x not in emoji.UNICODE_EMOJI])
    
    # corrects the bug that eliminates letters from words with accents
    text = ''.join(ch for ch in unicodedata.normalize('NFKD', text) 
    if not unicodedata.combining(ch))

    # removes special characters 
    if remove_mentions_hashtags:
        text = re.sub(r"@(\w+)", " ", text)
        text = re.sub(r"#(\w+)", " ", text)

    #
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    regex = re.compile('[' + re.escape(string.punctuation) + '0-9\\r\\t\\n]')
    nopunct = regex.sub(" ", text.lower())
    words = (''.join(nopunct)).split()
    
    # removes stopwords with less than two letters
    if(remove_stop):
        words = [w for w in words if w not in portuguese_stopwords]
        words = [w for w in words if len(w) > 2]  

    if(stem_words):
        stemmer = PorterStemmer()
        words = [stemmer.stem(w) for w in words]

    return list(words)


def tokenizacao(df):
    
    # gets the number of rows and columns from the dataframe
    rows, cols = df.shape

    # creates the dataset column with the vectorized text
    df['token'] = [preprocess_text(df["COMMENT_TEXT"][row]) for row in range(rows)]
    
    return df

# Topicos de classificação

- Cria lista de tópicos em ordem de prioridade que usaremos para a classificação dos comentários
- Criação do dicionário que associa os labels a seu tópico
- Tokeniza o  tópicos para usar na seleção dos labels

In [4]:
topics = [
'AQUECIMENTO', 
'ASSISTÊNCIA TÉCNICA', 
'ATENDIMENTO', 
'AUTO FALANTE', 
'BATERIA', 
'CÂMERA', 
'CARREGADOR', 
'CUSTO BENEFÍCIO', 
'DESIGN', 
'ENTREGA', 
'FLASH', 
'FONE', 
'JOGOS', 
'MEMÓRIA',
'PESO', 
'PREÇO', 
'PROCESSADOR', 
'QUALIDADE', 
'RESISTÊNCIA', 
'TAMANHO', 
'TELA', 
'TRAVAMENTO',
'VELOCIDADE', 
'GENÉRICO/OUTRO'
         ]

# localização do último tópico que não é "Genérico/Outro"
lasttopic = len(topics) - 1

# dicionário onde cada chave é uma categoria, e os valores são as palavras associadas a sua respectiva categoria 
dic_topics2 = {
'AQUECIMENTO' :'AQUECIMENTO' ,
'ASSISTÊNCIA TÉCNICA' :'ASSISTÊNCIA TÉCNICA' ,
'ATENDIMENTO' :'ATENDIMENTO' ,
'AUTO FALANTE' :'AUTO FALANTE' ,
'BATERIA' :'BATERIA' ,
'CÂMERA' :'CÂMERA' ,
'CARREGADOR' :'CARREGADOR' ,
'CUSTO BENEFÍCIO' :'CUSTO BENEFÍCIO' ,
'DESIGN' :'DESIGN' ,
'ENTREGA' :'ENTREGA' ,
'FLASH' :'FLASH' ,
'FONE' :'FONE' ,
'JOGOS' :'JOGOS' ,
'MEMÓRIA':'MEMÓRIA',
'PESO' :'PESO' ,
'PREÇO' :'PREÇO' ,
'PROCESSADOR' :'PROCESSADOR' ,
'QUALIDADE' :'QUALIDADE' ,
'RESISTÊNCIA' :'RESISTÊNCIA' ,
'TAMANHO' :'TAMANHO' ,
'TELA' :'TELA' ,
'TRAVAMENTO':'TRAVAMENTO',
'VELOCIDADE' :'VELOCIDADE' ,
'GENÉRICO/OUTRO':'GENÉRICO/OUTRO',         
}

# cria dicionário associndo um número de 0 a 23 a cada tópico na ordem da lista
dic_topics = {}

#
for n, item in enumerate(topics):
    dic_topics[n]=item
     
# tokeniza os tópicos da lista
topics_token = [preprocess_text(item) for item in topics]

# Modelo KNN Machine Learning

## Pipeline Data Prep

Tokeniza cada comentário na a coluna 'COMMENT_TEXT' do dataset e depois remove as linhas tokenizadas de tamanho 0

In [5]:
# escolhe as colunas que usaremos e remove as linhas nulas
df2 = preprocess_data(df, 
                      columns=['KEY','COMMENT_ID','COMMENT_TEXT'],
                      null = True
                     )

# reseta os indices e remove a coluna de indice
df2 = df2.reset_index().drop('index',axis=1)

# cria coluna com comentários tokenizados
df2 = tokenizacao(df2)

# creates an intermediary dataframe
df3 = df2.copy()

# remove as linhas que a tokenização ficou com tamanho 0 (sem texto)
for n, item in enumerate(df2.token):
    if len(item)==0:
        df3 = df3.drop(n)

## Primeiros labels 

- Criação dos 23 primeiros labels, onde são decididos vendo se o comentário possui a palavra do label, em ordem de prioridade da lista.

In [6]:
# defines empty list
lista_clusters = []

# cria clusters onde cada um deles possui comentários contendo um dos tópicos da lista, caso um comentário contenha dois
# dos tópicos, a ordem de prioridade é a mesma ordem na qual os tópicos estão na lista.
for n, objeto in enumerate(topics_token):
    
    if len(objeto) == 1:
        lista_1 = [item for item in df3.token if objeto[0] in item]
        for n, item in enumerate(lista_clusters):
            lista_1 = [item1 for item1 in lista_1 if item1 not in item]
        lista_clusters.append(lista_1)

    else:
        lista_2 = [item for item in df3.token if (
            objeto[0] in item) and (objeto[1] in item)]
        for n, item in enumerate(lista_clusters):
            lista_2 = [item1 for item1 in lista_2 if item1 not in item]
        lista_clusters.append(lista_2)

## Último Label

- Criação do último label, escolhendo os comentários no dataset que não possui nenhum dos 23 tópicos contidos no seu texto, sendo então caracterizado como "Genérico/outro"

- Temos uma lista onde cada elemento é uma lista com todas os comentários do dataset separados por seu label

- Criação de uma copia da lista, para fazer o balanceamento sem perder informações da lista de clusters original, caso ela seja necessário depois.

- Balanceamento para diminuir os 40 mil comentários na categoria "Genérico/outros" de maneira aleatória.

- Ao final conferimos o tamanho dessa lista, onde cada um desses tamanho é a quantidade de comentários em seu respectivo label.

In [7]:
temp = []

# junta todos os comentários de todos os tópicos em uma única lista sem divisão
for n, objeto in enumerate(lista_clusters):
    temp.extend(objeto)

# cria um último clusters formado por todos os comentários que não possui os elementos da lista acima, isto é não possui
# nenhum dos tópicos
ultimo_cluster = [item for item in df3.token if item not in temp]

# define esse último cluster como último elemento da lista de clusters, que vai servir como o tópico "genérico/outro"
lista_clusters[lasttopic] = ultimo_cluster

# cria uma copia da lista_clusters
lista_clusters1 = lista_clusters.copy()

# toma um sample de 2000 comentários para os clusters que tiveram mais que 3500 comentários, diminuindo o balanceamento
for n, item in enumerate(lista_clusters1):
    if len(item) > 3500:
        lista_clusters1[n] = random.sample(item, 2000)    

## Vectorization

 - definindo a função que usaremos para vetorizar os textos

In [8]:
# carrega as arquivo "pt_core_nes_md" do spacy, usado para vetorização de textos em português
nlp = spacy.load('pt_core_news_md')

# define a vetorização usando a função str.vector do spacy
def vec(s):
    return nlp.vocab[s].vector

## Vetorização - Coluna

 - Vetorizando toda a coluna de comentários, transformando cada uma das palavras em um vetor com 300 cordenadas e colocamos a média de cada uma dessas frases em uma matriz final.

In [9]:
# cria uma lista com a quantidade de elementos em cada um dos clusters
linhas = [len(item) for item in lista_clusters1]

# tamanho das palavras vetorizadas
vec_size = 300

# soma a quantidade de elementos de todos os clusters
rows = sum(linhas)

# cria uma lista vazia
list_of_matrix = [] 

# cria uma matriz vazia para ser preenchida com os comentários vetorizados ao final
final_feature_matrix = np.empty([rows, vec_size])

# cria preenche a matriz vazia criada acima com os comentários vetorizados
for n, item in enumerate(lista_clusters1):
    for corpus in item: 
        matrix = np.empty([len(corpus), vec_size]) 
                                              
        for idx, word in enumerate(corpus):
            matrix[idx,:] = vec(word) 
        list_of_matrix.append(matrix)

# cama comentário vetorizado possui 300 variáveis tiramos a média de cada palavra em cada frase para preencher a matriz
# criada acima e preenche-la com esses vetores
for row in range(rows):
    final_feature_matrix[row,:] = list_of_matrix[row].mean(axis = 0)

## Labels e Shuffle

- Adicionando a matriz vetorizada, uma coluna com labels, baseada no tamanho de cada uma das listas contidas na "lista_clusters1"

- Fazendo um shuffle na matriz para diminuir as chances de viés no treinamento do modelo

In [10]:
#
labels = []

# criamos uma lista com os labels de 0 a 23 para cada tópico
for n, item in enumerate(lista_clusters1):
    for i, objeto in enumerate(item):
        labels.append(n)
    
# transformamos em um array
x = np.array(labels)

# fazemos o reshape para concatenar a matriz de variáveis
x = x.reshape(-1,1)

#  concatenamos a matriz de variáveis com os labeis
final_matrix = np.concatenate((final_feature_matrix, x), axis=1)

# fazemos um shuffle para os comentários não ficar na sequencia pelos labels
final_matrix1 = shuffle(final_matrix, random_state = 42)

## Treinamento e predição

* Dividindo os dados em treino e teste usando a função train_test_split do kslearn
* Treinando um knn classificador com 2 vizinhos
* Fazendo a predição nos dados de teste
* Conferindo acurácia

In [11]:
# dividindo em treino e teste
Xtreino, Xteste, ytreino, yteste = train_test_split(
    final_matrix1[:, 0:-1], final_matrix1[:, -1], train_size = 0.7, random_state=42)

# definindo o knn com 2 vizinhos
knn = KNeighborsClassifier(n_neighbors = 2)

# treinando o knn
knn.fit(Xtreino, ytreino)

# fazendo a predição
pred = knn.predict(Xteste)

# vendo a acurácia
accuracy_score(yteste,pred)

0.6815968841285297

## Gera dataframe com tópicos usando KNN

Função pra classificar comentários de um dataset usando o KNN treinado acima.

In [12]:
def classifica_text_knn(string):
    try:
        
        # preprocessamento criando a matriz com cada palavra da frase vetorizada
        textoProcessado = preprocess_text(string)
        matrix = np.empty([len(textoProcessado), 300])
        for idx, word in enumerate(textoProcessado):
            matrix[idx,:] = vec(word)

        # matriz final com 300 colunas
        final_feature_matrix = np.empty([1, 300])

        # cala palabra da string vetorizada possui 300 variáveis, 
        # tiramos a média de cada palavra na string para preencher a matriz
        # criada acima e preenche-la com esses vetores
        final_feature_matrix = matrix.mean(axis = 0).reshape(1,-1)

        # predição usando o Knn
        knn.predict(final_feature_matrix)

        # classificação usando o dicionário que relaciona cada número à uma categoria
        classi = dic_topics[knn.predict(final_feature_matrix)[0]]
    
    # caso a sting seja vazia, como um "NaN", retorna a categoria 'Genérico/Outro
    except ValueError as e:
        classi = 'Genérico/Outro'
      
    # caso a atring vetorizada fique sem nenhum comprimento, exemplo: strings que são formadas apenas por emotions
    # também são classificadas como 'Genérico/Outro
    except TypeError as e:
        
        classi = 'Genérico/Outro'
    return classi

# função para classificar um dataset usando o Knn e a função criada acima
def classifca_dataframe_knn(df):
    
    # cria uma copia do dataset
    df1 = df.copy()
    
    # aplica a função "classifica_text_knn" em todos os comentários do dataset
    df1['categoria_knn'] = df.COMMENT_TEXT.apply(classifica_text_knn)
        
    return df1

## Pipeline Treina Modelo

In [13]:
df = classifca_dataframe_knn(df)

# Gera dataframe com tópicos usando regex

- Função para classificar os comentários de um dataframe usando regex.
- Função atualizada: funciona para qualquer quantidade de tópicos e qualquer quantidade de palavras relacionadas a cada tópico.

## Funções do Modelo de Regex

Comentar o código passo a passo

In [14]:
def classifica_dataframe(df):
    '''
    A função recebe um dataset que possui uma coluna de textos,
    todas as frases que possui alguma palavra relacionada ao tópico é classficada como aquele tópico,
    frases que não contém palavras de nenhum tópico são classificadas como 'Genérico/Outro',
    frases com comentários nulos também são classificadas como 'Genérico/Outro'.
    '''
    
    if len(df)==0:
        
        return df
    
    else:
        
        # preprocessamento removendo linhas nulas
        df1 = preprocess_data(df, 
                          columns=['KEY',
                                   'COMMENT_ID',
                                   'COMMENT_TEXT'
                                  ],
                          null = True
                             )
        
        # removendo as linhas que não são nulas, assim conseguimos somente as linhas nulas 
        # para classifica-las como 'Genérico/Outro' 
        dfdelta = df.drop(df1.index)
        
        # classificando as linhas nulas como 'Genérico/Outro'
        dfdelta['categoria'] = 'Genérico/Outro'
        
        # ordenado as colunas para concatenação no final
        dfdelta = dfdelta[['COMMENT_TEXT',
                           'KEY',
                           'categoria'
                          ]]
        
        # renomeando a coluna de comentários para 'token', assim conseguimos fazer uma concatenação futura de maneira
        # mais facil, ao final, não iremos utilizar as colunas desse dataset criado aqui
        dfdelta = dfdelta.rename(columns = {'COMMENT_TEXT':'token'})

        # resetando indice do df preprocessado
        df1 = df1.reset_index().drop('index',axis=1)
        
        # tokenizando a coluna de texto
        df1 = tokenizacao(df1)

        # criando lista vazia para adicionar os indices que o a tokenização resultou em uma lista vazia
        linhas_vazias = []
        
        # criando copia do df1
        df2 = df1.copy()

        # removendo as linhas vazias e adicionando o indice na lista vazia criada acima
        for n, item in enumerate(df2.token):
            if len(item)==0:
                df2 = df2.drop(n)
                linhas_vazias.append(n)

        # resetando os indices
        df2 = df2.reset_index().drop(columns={'index'})
        
        # criando lista vazia para os clusters
        lista_clusters = []

        # salvando um dataset somente com as linhas vazias para classficica-las como 'Genérico/Outro'
        df_lv = df1.loc[linhas_vazias,:]

        # classificando as linhas vazias como 'Genérico/Outro'
        df_lv['categoria'] = 'Genérico/Outro'
        
        # tomando somente as colunas ['token','KEY','categoria']
        df_lv = df_lv[['token','KEY','categoria']]

        # criando os clusters de acordo com as palavras de cada tópicos, frases que contem essas palavras, 
        # são classificadas com o tópico relacionado e adicionado a lista_clusters
        for n, objeto in enumerate(topics_token):
            lista_1 = []
            for i in range(len(df2)):
                inter = set(objeto).intersection(df2.token[i])
                if len(inter)!=0:
                    lista_1.append([df2.token[i], df2.KEY[i]])
            for n, item in enumerate(lista_clusters):
                lista_1 = [item1 for item1 in lista_1 if item1 not in item]
            lista_clusters.append(lista_1)

        # criando lista vazia temporaria para criação do último cluster relacionado a categoria 'Genérico/Outro'
        temp = []
        
        # juntando todas as frases que foram classificadas com algum dos tópicos em uma única lista
        for n, objeto in enumerate(lista_clusters[:lasttopic]):
            lista_temp = [item[0] for item in objeto] 
            temp.extend(lista_temp)
        ultimo_cluster = []
        
        # criando último clusters como sendo todas as frases que não foram classificadas como nenhum dos 
        # tópicos anteriores
        for i in range(len(df2)):
            if (df2.token[i] not in temp):
                ultimo_cluster.append([df2.token[i], df2.KEY[i]])

        # adicionando o último cluster na lista_cluster
        lista_clusters[lasttopic] = ultimo_cluster

        # criando lista vazia para adicionar os labels
        labels = []

        # criando os labels pela ordem da lista
        for n, item in enumerate(lista_clusters):
            for i, objeto in enumerate(item):
                labels.append(n)

        # relacionando cada label a sua categoria em uma lista
        categorias = [dic_topics[item] for item in labels]

        # criando o dataset com as frases relacionadas ao primeiro tópico
        dff = pd.DataFrame(lista_clusters[0])
        
        # concatenando o resto das frases no dataset criado acima
        for n, objeto in enumerate(lista_clusters[1:]):
            dff = pd.concat([dff,pd.DataFrame(objeto)])

        # renomeando as colunas para concatenação 
        dff = dff.rename(columns={0:'token',1:'KEY'})

        # criando coluna de categorias usando a lista criada 'categorias'
        dff['categoria'] = categorias

        # concatenando com o dataset de linhas vazias que criamos no começo
        dff = pd.concat([dff,df_lv])
        
        # concatenando com o dataset de linhas nulas que criamos também no começo
        dff = pd.concat([dff, dfdelta])

        # fazendo o merge do dataset original com esse dataset que possui as 'KEYS', e as categorias
        df_final = pd.merge(df,dff, on='KEY', how='left')
        
        # esconhendo somente as colunas originais do dataset mais a coluna de categorias
        df_final = df_final[df.columns.tolist()+['categoria']]

        return df_final

## Pipeline do Modelo de Regex

In [15]:
# chama a função que cria o dataframe final
df = classifica_dataframe(df)

# mostra o dataframe final
df

,Unnamed: 0,KEY,AP2,COLLECT_DATE,COMMENT_DATE,COMMENT_DISLIKE,COMMENT_ID,COMMENT_LIKE,COMMENT_TEXT,COMMENT_TITLE,...,COUNTRY,LAST_CRAWLER_DATE,PROD_BRAND,PROD_DESC,PROD_HREF,PRODUCT_TYPE,RATING_VALUE,SITE,categoria_knn,categoria
0,0,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcn...,SEDA,2021-10-12,06/10/21,0,9934083.0,0,Muito bom,NaN,...,BR,2021-10-12 03:50,SAMSUNG,"Smart TV LED 43\"" Full HD Samsung T5300 com HD...",https://www.casasbahia.com.br/smart-tv-led-43-...,CE,5.0,https://www.casasbahia.com.br/,GENÉRICO/OUTRO,GENÉRICO/OUTRO
1,1,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcn...,SEDA,2021-10-12,29/09/21,0,9874011.0,0,Muito bom,NaN,...,BR,2021-10-12 03:50,LG,"Smart TV LED 32\"" HD LG 32LM625BPSB ThinQ AI I...",https://www.casasbahia.com.br/smart-tv-led-32-...,CE,5.0,https://www.casasbahia.com.br/,GENÉRICO/OUTRO,GENÉRICO/OUTRO
2,2,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcn...,SEDA,2021-10-12,08/09/21,0,9709465.0,0,Muito bom,NaN,...,BR,2021-10-12 03:50,LG,"Smart TV LED 32\"" HD LG 32LM625BPSB ThinQ AI I...",https://www.casasbahia.com.br/smart-tv-led-32-...,CE,5.0,https://www.casasbahia.com.br/,GENÉRICO/OUTRO,GENÉRICO/OUTRO
3,3,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcn...,SEDA,2021-10-12,19/08/21,0,9548553.0,0,Muito bom,NaN,...,BR,2021-10-12 03:51,LG,"Smart TV LED 32\"" HD LG 32LM625BPSB ThinQ AI I...",https://www.casasbahia.com.br/smart-tv-led-32-...,CE,5.0,https://www.casasbahia.com.br/,GENÉRICO/OUTRO,GENÉRICO/OUTRO
4,4,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcn...,SEDA,2021-10-12,17/08/21,0,9528145.0,0,Muito bom,NaN,...,BR,2021-10-12 03:51,LG,"Smart TV LED 32\"" HD LG 32LM625BPSB ThinQ AI I...",https://www.casasbahia.com.br/smart-tv-led-32-...,CE,5.0,https://www.casasbahia.com.br/,GENÉRICO/OUTRO,GENÉRICO/OUTRO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7145,7145,aHR0cHM6Ly93d3cucG9udG9mcmlvLmNvbS5ici9zYW1zdW...,SEDA,2021-10-16,13/10/21,0,9978117.0,0,Não nos avisaram a data e o horário da entrega...,NaN,...,BR,2021-10-16 07:38,SAMSUNG,Samsung Smart TV 60&quot; Crystal UHD 4K 60AU8...,https://www.pontofrio.com.br/samsung-smart-tv-...,CE,5.0,https://www.pontofrio.com.br/,ENTREGA,ENTREGA
7146,7146,aHR0cHM6Ly93d3cucG9udG9mcmlvLmNvbS5ici9zbWFydC...,SEDA,2021-10-16,16/10/21,0,9999136.0,0,Excelente imagem e acabamento. Sobre funcional...,NaN,...,BR,2021-10-16 08:30,SAMSUNG,Smart TV Crystal 4K 55” Samsung UN55AU7700GXZD...,https://www.pontofrio.com.br/smart-tv-crystal-...,CE,5.0,https://www.pontofrio.com.br/,TELA,GENÉRICO/OUTRO
7147,7147,aHR0cHM6Ly93d3cucG9udG9mcmlvLmNvbS5ici9zbWFydC...,SEDA,2021-10-16,04/10/21,0,9909097.0,0,Fiz um teste rápido e gostei. Só não dei 5 est...,NaN,...,BR,2021-10-16 09:24,SAMSUNG,Smart TV LED 43” Full HD Samsung LH43BET com H...,https://www.pontofrio.com.br/smart-tv-led-43-f...,CE,4.0,https://www.pontofrio.com.br/,GENÉRICO/OUTRO,GENÉRICO/OUTRO
7148,7148,aHR0cHM6Ly93d3cucG9udG9mcmlvLmNvbS5ici9zbWFydC...,SEDA,2021-10-16,17/09/21,0,9777657.0,0,Excelente,NaN,...,BR,2021-10-16 09:24,SAMSUNG,Smart TV LED 43” Full HD Samsung LH43BET com H...,https://www.pontofrio.com.br/smart-tv-led-43-f...,CE,5.0,https://www.pontofrio.com.br/,GENÉRICO/OUTRO,GENÉRICO/OUTRO
